# S4 · Ejercicio 5c — Inferencia: predecir por coordenada

**Curso de Big Data · Dr. Abel Coronado** · Semana 4

> ⚙️ **Esta variante es para: Container (Podman) / Vagrant — Spark 4.0 · Scala 2.13.**  
> Comprueba tu versión con `import pyspark; pyspark.__version__`. Si NO coincide, usa la carpeta de tu plataforma (`container/` o `portable/`).

🎯 **Objetivo.** Cargar el modelo y predecir OXXO vs Abarrotes para una coordenada nueva, recalculando sus features en vivo.

📦 **Datos.** Modelo `gbt_oxxo_model` + censo/denue en HDFS.

✅ **Prerrequisitos.** Ejercicio 5b completado. Requiere pyproj.

📝 **Entregable.** Predicción (clase + probabilidad) para al menos 2 coordenadas.

⬇️ **Descarga (Hugging Face):** https://huggingface.co/datasets/abxda/bdp-lab/resolve/main/cuadernos/semana_4/container/04_PrediccionPorCoordenada.ipynb

💬 **¿Un error?** Captura (celda + mensaje) y repórtalo por **Blackboard** (indica SO y paso).

---

> Lee los comentarios de cada celda: explican el porqué de cada paso.


In [ ]:
# =============================================================================
# Celda 1: Inicialización de Spark y Sedona
# =============================================================================
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, expr, lit, sum as _sum
from sedona.spark import SedonaContext

# Detener cualquier sesión previa para una inicialización limpia
try:
    spark.stop()
    print("Sesión de Spark anterior detenida.")
except:
    pass

print("» Paso 1: Configurando la sesión de Spark para un procesamiento geoespacial...")
spark = (
    SparkSession.builder.appName("PrediccionOXXO_HDFS")
    .config("spark.driver.memory", "2g")  # BDP v7: recorte de un estado
    .config("spark.locality.wait", "0")
    # BDP v7: Sedona 1.9.1 ya viene pre-cargado en el laboratorio (sin descargas de Maven)
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "org.apache.sedona.core.serde.SedonaKryoRegistrator")
    .config("spark.sql.extensions", "org.apache.sedona.sql.SedonaSqlExtensions")
    .config("spark.sql.ansi.enabled", "false").getOrCreate()
)

sedona = SedonaContext.create(spark)
print("✅ Sesión de Spark con Apache Sedona iniciada correctamente.")

In [ ]:
# =============================================================================
# Celda 2: Carga de Datos y Modelo
# =============================================================================
from pyspark.ml.classification import GBTClassificationModel
from pyspark.sql.functions import col, lower, substring, when
from pyspark.sql.types import IntegerType, DoubleType, LongType, FloatType

print("\n» Paso 2: Cargando recursos necesarios desde HDFS.")

# --- 2.1 Definir las rutas en HDFS ---
CENSO_PATH_HDFS = "hdfs:///data/raw/geodatos_mexico/censo_2020_nacional.parquet"
DENUE_PATH_HDFS = "hdfs:///data/raw/geodatos_mexico/denue_nacional.parquet"
MODEL_PATH_HDFS = "hdfs:///models/gbt_oxxo_model"

# --- 2.2 Cargar DataFrames ---
print(f"   Cargando Censo desde: {CENSO_PATH_HDFS}")
censo_sdf = spark.read.format("geoparquet").load(CENSO_PATH_HDFS)

print(f"   Cargando DENUE desde: {DENUE_PATH_HDFS}")
denue_raw_sdf = spark.read.format("geoparquet").load(DENUE_PATH_HDFS)

# --- 2.3 Cargar el Modelo GBT ---
print(f"   Cargando modelo GBT desde: {MODEL_PATH_HDFS}")
gbt_model = GBTClassificationModel.load(MODEL_PATH_HDFS)
print("✅ Modelo cargado exitosamente.")

# --- 2.4 Preprocesamiento de DENUE y Censo (IDÉNTICO AL DEL ENTRENAMIENTO) ---
# Es crucial replicar exactamente el mismo pre-procesamiento.
print("   Pre-procesando DENUE y Censo...")
denue_sdf = denue_raw_sdf.withColumn("codigo_act_2c", substring(col("codigo_act"), 1, 2)) \
    .withColumn("est_per_ocu",
        when(col("per_ocu").contains("0 a 5"), 2.5)
        .when(col("per_ocu").contains("6 a 10"), 8.0)
        .when(col("per_ocu").contains("11 a 30"), 20.5)
        .when(col("per_ocu").contains("31 a 50"), 40.5)
        .when(col("per_ocu").contains("51 a 100"), 75.5)
        .when(col("per_ocu").contains("101 a 250"), 175.5)
        .when(col("per_ocu").contains("251 y más"), 350.0)
        .otherwise(1.0)
    )

numeric_censo_cols = [f.name for f in censo_sdf.schema.fields if isinstance(f.dataType, (IntegerType, DoubleType, LongType, FloatType))]
censo_sdf = censo_sdf.na.fill(0, subset=numeric_censo_cols)

print("✅ Recursos cargados y pre-procesados.")

In [ ]:
# =============================================================================
# Celda 3: Función de Inferencia (Versión Corregida y Robusta)
# =============================================================================
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import lit
import pyproj # Asegúrate de que esta librería está disponible

# --- Copiamos las mismas funciones de agregación (sin cambios) ---
def aggregate_denue_features(base_df, denue_df, buffer_col_name, suffix):
    joined = base_df.join(denue_df.alias("d"), expr(f"ST_Intersects({buffer_col_name}, d.geometry)"))
    agg_epo = joined.groupBy(base_df.id).agg(_sum("d.est_per_ocu").alias(f"epo_{suffix}"))
    act_codes = ['51', '54', '11', '22', '52', '71', '43', '31', '61', '46', '23', '55', '93', '53', '81', '33', '48', '32', '56', '49', '62', '21', '72']
    agg_pivot = joined.groupBy(base_df.id).pivot("d.codigo_act_2c", act_codes).agg(_sum("d.est_per_ocu"))
    for code in act_codes:
        agg_pivot = agg_pivot.withColumnRenamed(code, f"act_{code}_{suffix}")
    return agg_epo.join(agg_pivot, "id", "outer")

def aggregate_censo_features(base_df, censo_df, buffer_col_name, suffix):
    joined = base_df.join(censo_df.alias("c"), expr(f"ST_Intersects({buffer_col_name}, c.geometry)"))
    numeric_censo_cols = [f.name for f in censo_df.schema.fields if isinstance(f.dataType, (IntegerType, DoubleType, LongType, FloatType))]
    agg_exprs = [_sum(f"c.{col}").alias(f"censo_{col}_{suffix}") for col in numeric_censo_cols]
    return joined.groupBy(base_df.id).agg(*agg_exprs)

# --- Función de conversión de coordenadas ---
def convertir_4326_a_6372(lon, lat):
    crs_origen = pyproj.CRS("EPSG:4326")
    crs_destino = pyproj.CRS("EPSG:6372")
    transformador = pyproj.Transformer.from_crs(crs_origen, crs_destino, always_xy=True)
    x_6372, y_6372 = transformador.transform(lon, lat)
    return (x_6372, y_6372)


# --- Función principal para la predicción (CORREGIDA) ---
def predecir_potencial_tienda(lat, lon):
    """
    Calcula todas las features para una coordenada dada, usa el modelo cargado
    para predecir y devuelve el resultado.
    """
    print(f"\nCalculando features para la coordenada (Lat: {lat}, Lon: {lon})...")

    # 1. Cargar la lista de features EXACTAS usadas en el entrenamiento
    FEATURES_LIST_PATH = "hdfs:///models/gbt_oxxo_model_features"
    training_feature_cols = spark.read.text(FEATURES_LIST_PATH).rdd.map(lambda row: row.value).collect()
    print(f"   Se usarán {len(training_feature_cols)} features consistentes con el modelo.")

    # 2. Convertir coordenadas y crear DataFrame inicial
    x_proj, y_proj = convertir_4326_a_6372(lon, lat)
    point_df = spark.createDataFrame([(1, x_proj, y_proj)], ["id", "x", "y"]) \
                    .withColumn("geometry", expr("ST_Point(x, y)"))

    # 3. Crear buffers y calcular features agregadas (igual que antes)
    point_with_buffers = point_df.withColumn("buffer_100m", expr("ST_Buffer(geometry, 100)")).withColumn("buffer_500m", expr("ST_Buffer(geometry, 500)"))
    denue_agg_100 = aggregate_denue_features(point_with_buffers, denue_sdf, "buffer_100m", "100")
    denue_agg_500 = aggregate_denue_features(point_with_buffers, denue_sdf, "buffer_500m", "500")
    censo_agg_100 = aggregate_censo_features(point_with_buffers, censo_sdf, "buffer_100m", "100")
    censo_agg_500 = aggregate_censo_features(point_with_buffers, censo_sdf, "buffer_500m", "500")

    # 4. Unir todo
    features_df = point_df.select("id") \
        .join(denue_agg_100, "id", "left_outer") \
        .join(denue_agg_500, "id", "left_outer") \
        .join(censo_agg_100, "id", "left_outer") \
        .join(censo_agg_500, "id", "left_outer") \
        .na.fill(0)

    # 5. *** ASEGURAR TODAS LAS COLUMNAS ***
    # Iterar sobre la lista de features del entrenamiento. Si una columna no existe
    # en nuestro DataFrame de predicción, la añadimos con valor 0.
    existing_cols = features_df.columns
    for col_name in training_feature_cols:
        if col_name not in existing_cols:
            features_df = features_df.withColumn(col_name, lit(0))
    # 6. Ensamblar el vector de features usando la lista del entrenamiento
    assembler = VectorAssembler(inputCols=training_feature_cols, outputCol="features", handleInvalid="skip")
    final_df = assembler.transform(features_df)

    # 7. Realizar la predicción
    print("   Realizando predicción con el modelo GBT...")
    prediction_result = gbt_model.transform(final_df)

    # 8. Extraer y devolver el resultado
    result_row = prediction_result.select("prediction", "probability").first()
    prediction = result_row['prediction']
    probability = result_row['probability']
    clase = "OXXO" if prediction == 1.0 else "Abarrotes"
    confianza = probability[int(prediction)] * 100
    
    return clase, confianza

print("✅ Función de inferencia (con consistencia de features) definida.")

In [ ]:
#python -m pip install pyproj

In [ ]:
import pyproj

def convertir_4326_a_6372(lon, lat):
    # Definir los sistemas de coordenadas de origen (WGS84) y destino (ITRF2008 / Mexico)
    crs_origen = pyproj.CRS("EPSG:4326")
    crs_destino = pyproj.CRS("EPSG:6372")
    # Crear el objeto para transformar entre los dos sistemas
    # always_xy=True asegura que el orden de entrada es (longitud, latitud)
    transformador = pyproj.Transformer.from_crs(crs_origen, crs_destino, always_xy=True)
    # Realizar la transformación
    x_6372, y_6372 = transformador.transform(lon, lat)
    return (x_6372, y_6372)

In [ ]:
# =============================================================================
# Celda 4: Probar la Predicción
# =============================================================================

# --- Define las coordenadas de la nueva ubicación a evaluar ---
# Ejemplo 1: Una zona residencial en la Ciudad de México
latitud_usuario = 19.3541
longitud_usuario = -99.1687
clase_predicha, confianza_prediccion = predecir_potencial_tienda(latitud_usuario, longitud_usuario)

# --- Mostrar el resultado de forma clara ---
print("\n" + "="*50)
print("              RESULTADO DE LA PREDICCIÓN")
print("="*50)
print(f"Coordenadas       : Lat {latitud_usuario}, Lon {longitud_usuario}")
print(f"Potencial sugerido: {clase_predicha}")
print(f"Confianza         : {confianza_prediccion:.2f}%")
print("="*50)

In [ ]:
# =============================================================================
# Celda 4: Probar la Predicción
# =============================================================================
 
# --- Define las coordenadas de la nueva ubicación a evaluar ---
# Ejemplo 1: Una zona residencial en la Aguascalientes
latitud_usuario = 21.936843851728213
longitud_usuario = -102.2605316444216
clase_predicha, confianza_prediccion = predecir_potencial_tienda(latitud_usuario, longitud_usuario)

# --- Mostrar el resultado de forma clara ---
print("\n" + "="*50)
print("              RESULTADO DE LA PREDICCIÓN")
print("="*50)
print(f"Coordenadas       : Lat {latitud_usuario}, Lon {longitud_usuario}")
print(f"Potencial sugerido: {clase_predicha}")
print(f"Confianza         : {confianza_prediccion:.2f}%")
print("="*50)

In [ ]:

# =============================================================================
# Celda 4: Probar la Predicción
# =============================================================================
 
# --- Define las coordenadas de la nueva ubicación a evaluar ---
# Ejemplo 1: Una zona residencial en la Aguascalientes
latitud_usuario = 20.6496963691001
longitud_usuario = -103.26470786900687
clase_predicha, confianza_prediccion = predecir_potencial_tienda(latitud_usuario, longitud_usuario)

# --- Mostrar el resultado de forma clara ---
print("\n" + "="*50)
print("              RESULTADO DE LA PREDICCIÓN")
print("="*50)
print(f"Coordenadas       : Lat {latitud_usuario}, Lon {longitud_usuario}")
print(f"Potencial sugerido: {clase_predicha}")
print(f"Confianza         : {confianza_prediccion:.2f}%")
print("="*50)

In [ ]:
#curl -X POST http://localhost:5001/predict -H "Content-Type: application/json" -d "{\"latitude\": 20.6496963691001, \"longitude\": -103.26470786900687}"

In [ ]:

# =============================================================================
# Celda 4: Probar la Predicción
# =============================================================================
 
# --- Define las coordenadas de la nueva ubicación a evaluar ---
#Primer anillo Santa Anita Avenida importante
latitud_usuario = 21.894639049283427
longitud_usuario = -102.27747206999581
clase_predicha, confianza_prediccion = predecir_potencial_tienda(latitud_usuario, longitud_usuario)

# --- Mostrar el resultado de forma clara ---
print("\n" + "="*50)
print("              RESULTADO DE LA PREDICCIÓN")
print("="*50)
print(f"Coordenadas       : Lat {latitud_usuario}, Lon {longitud_usuario}")
print(f"Potencial sugerido: {clase_predicha}")
print(f"Confianza         : {confianza_prediccion:.2f}%")
print("="*50)

## 💡 Mejora futura (nota para ti, alumno)

Este modelo **recalcula** las features espaciales (buffers + spatial joins contra el DENUE y el Censo NACIONAL) **cada vez** que predices un punto. Por eso la inferencia es lenta y necesita ~6 GB de memoria de driver.

En un sistema de **producción** se haría distinto: se **pre-calculan UNA sola vez** las features de una malla o zona y se guardan (en HDFS o una base de datos); la predicción solo busca el punto más cercano → responde en **milisegundos** en vez de minutos, y sin tanta memoria.

> Piensa cómo lo rediseñarías: ese es el salto de *“notebook que funciona”* a *“servicio que escala”*. (Mejora propuesta para una versión futura del curso.)
